# Cross-Encoder Reranking

## Objective

The retrieval stage reduces the 10,000-resume search space to a smaller
candidate pool.

In this stage, a cross-encoder reranker evaluates each
(job description, candidate resume) pair jointly and reorders the
retrieved candidates using deeper semantic interactions.

Pipeline:

Job Description
      ↓
Skill-Aware BM25
      ↓
Top-100 candidate resumes
      ↓
Qwen3-Reranker-0.6B
      ↓
Reranked candidates

The experiment begins with a latency and GPU-memory benchmark on a small
sample before scaling to the full candidate-retrieval benchmark.

In [4]:
import pandas as pd
import numpy as np
import torch

# Load dataset
resumes_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/resumes/train-00000-of-00001.parquet"
)

jobs_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/jobs/train-00000-of-00001.parquet"
)

matches_df = pd.read_parquet(
    "https://huggingface.co/datasets/michaelozon/candidate-matching-synthetic/resolve/main/matches/train-00000-of-00001.parquet"
)

# Helper
def list_to_text(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return " ".join(str(item) for item in value)
    return str(value)


# Rebuild retrieval text
def build_job_text(job):
    return " ".join([
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
        f"Description: {job['description']}",
        f"Requirements: {list_to_text(job['requirements'])}",
    ])


def build_resume_text(resume):
    return " ".join([
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Years of experience: {resume['years_experience']}",
        f"Industry: {resume['industry']}",
        f"Education: {resume['education']}",
        f"Skills: {list_to_text(resume['skills'])}",
        f"Summary: {resume['summary']}",
        f"Experience: {list_to_text(resume['experience_bullets'])}",
    ])


resumes_df["retrieval_text"] = resumes_df.apply(
    build_resume_text, axis=1
)

jobs_df["retrieval_text"] = jobs_df.apply(
    build_job_text, axis=1
)

resume_ids = resumes_df["resume_id"].tolist()

# Load previously generated Qwen embeddings
qwen_resume_embeddings = np.load(
    "data/qwen3_embedding_resume_embeddings.npy"
)

qwen_job_embeddings_query = np.load(
    "data/qwen3_embedding_job_embeddings_query.npy"
)

print("Resumes:", len(resumes_df))
print("Jobs:", len(jobs_df))
print("Matches:", len(matches_df))
print("Qwen resume embeddings:", qwen_resume_embeddings.shape)
print("Qwen job embeddings:", qwen_job_embeddings_query.shape)

Resumes: 10000
Jobs: 2500
Matches: 2500
Qwen resume embeddings: (10000, 1024)
Qwen job embeddings: (2500, 1024)


In [1]:
import torch
from sentence_transformers import CrossEncoder

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda"
)

print("Reranker loaded successfully.")

PyTorch: 2.14.0+cu130
CUDA available: True


modules.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

C:\Users\USER\OneDrive\Desktop\TANISHA_PRAKASH\SkillAlign\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--Qwen--Qwen3-Reranker-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/741 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

Default prompt name is set to 'query'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Reranker loaded successfully.


In [2]:
print("Device:", reranker.model.device)
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda:0
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [5]:
import time

sample_job = jobs_df.iloc[0]["retrieval_text"]
sample_resume_texts = resumes_df["retrieval_text"].iloc[:100].tolist()

pairs = [
    (sample_job, resume_text)
    for resume_text in sample_resume_texts
]

start = time.time()

sample_scores = reranker.predict(
    pairs,
    batch_size=4,
    show_progress_bar=True
)

elapsed = time.time() - start

print("Scores shape:", sample_scores.shape)
print(f"Time: {elapsed:.2f}s")
print(f"Pairs/sec: {100 / elapsed:.2f}")

Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Scores shape: (100,)
Time: 8.40s
Pairs/sec: 11.90


In [6]:
import time

start = time.time()

sample_scores_b8 = reranker.predict(
    pairs,
    batch_size=8,
    show_progress_bar=True
)

elapsed_b8 = time.time() - start

print("Scores shape:", sample_scores_b8.shape)
print(f"Time: {elapsed_b8:.2f}s")
print(f"Pairs/sec: {100 / elapsed_b8:.2f}")
print(f"Estimated 250k pairs: {(250000 / (100 / elapsed_b8)) / 3600:.2f} hours")

Batches:   0%|          | 0/13 [00:00<?, ?it/s]

Scores shape: (100,)
Time: 6.66s
Pairs/sec: 15.01
Estimated 250k pairs: 4.63 hours


In [7]:
import time

start = time.time()

sample_scores_b16 = reranker.predict(
    pairs,
    batch_size=16,
    show_progress_bar=True
)

elapsed_b16 = time.time() - start

print("Scores shape:", sample_scores_b16.shape)
print(f"Time: {elapsed_b16:.2f}s")
print(f"Pairs/sec: {100 / elapsed_b16:.2f}")
print(
    f"Estimated 250k pairs: "
    f"{(250000 / (100 / elapsed_b16)) / 3600:.2f} hours"
)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Scores shape: (100,)
Time: 6.35s
Pairs/sec: 15.75
Estimated 250k pairs: 4.41 hours


In [8]:
from sentence_transformers import CrossEncoder

reranker_256 = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda",
    max_length=256
)

print("Reranker 256 loaded.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Reranker 256 loaded.


In [9]:
import time

start = time.time()

sample_scores_256 = reranker_256.predict(
    pairs,
    batch_size=16,
    show_progress_bar=True
)

elapsed_256 = time.time() - start

print("Scores shape:", sample_scores_256.shape)
print(f"Time: {elapsed_256:.2f}s")
print(f"Pairs/sec: {100 / elapsed_256:.2f}")
print(
    f"Estimated 250k pairs: "
    f"{(250000 / (100 / elapsed_256)) / 3600:.2f} hours"
)

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Scores shape: (100,)
Time: 5.25s
Pairs/sec: 19.04
Estimated 250k pairs: 3.65 hours


In [10]:
from scipy.stats import spearmanr

corr, p_value = spearmanr(
    sample_scores,
    sample_scores_256
)

print(f"Spearman rank correlation: {corr:.4f}")
print(f"p-value: {p_value:.4g}")

# Compare Top-10 overlap
top10_512 = set(np.argsort(-sample_scores)[:10])
top10_256 = set(np.argsort(-sample_scores_256)[:10])

top10_overlap = len(top10_512 & top10_256) / 10

print(f"Top-10 overlap: {top10_overlap:.2%}")

Spearman rank correlation: 0.9803
p-value: 6.976e-71
Top-10 overlap: 90.00%


## Sequence-Length Ablation

The Qwen3-Reranker-0.6B was benchmarked with two maximum sequence
length configurations.

The 256-token configuration achieved higher throughput while maintaining
very high agreement with the longer-input configuration:

- Spearman rank correlation: 0.9803
- Top-10 ranking overlap: 90%

Based on this configuration-level benchmark, max_length=256 was selected
for the subsequent reranking experiments.

This result is based on a 100-pair benchmark and is used as an efficiency
ablation rather than a final model-quality evaluation.

In [11]:
# Reproducible Day 4 development evaluation set

jobs_eval = (
    jobs_df
    .sample(n=250, random_state=42)
    .reset_index(drop=True)
)

print("Evaluation jobs:", len(jobs_eval))
print("Expected reranker pairs:", len(jobs_eval) * 100)

Evaluation jobs: 250
Expected reranker pairs: 25000


In [12]:
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import csr_matrix
import numpy as np

def build_skill_aware_job_text(job):
    return " ".join([
        f"Job title: {job['job_title']}",
        f"Seniority: {job['seniority']}",
        f"Industry: {job['industry']}",
        f"Must-have skills: {list_to_text(job['must_have_skills'])}",
        f"Nice-to-have skills: {list_to_text(job['nice_to_have_skills'])}",
    ])


def build_skill_aware_resume_text(resume):
    return " ".join([
        f"Role: {resume['role']}",
        f"Seniority: {resume['seniority']}",
        f"Industry: {resume['industry']}",
        f"Skills: {list_to_text(resume['skills'])}",
    ])


resumes_df["skill_aware_text"] = resumes_df.apply(
    build_skill_aware_resume_text,
    axis=1
)

jobs_df["skill_aware_text"] = jobs_df.apply(
    build_skill_aware_job_text,
    axis=1
)

skill_vectorizer = CountVectorizer(
    lowercase=True,
    tokenizer=lambda x: x.lower().split(),
    preprocessor=None,
    token_pattern=None
)

X_skill = skill_vectorizer.fit_transform(
    resumes_df["skill_aware_text"]
)

N = X_skill.shape[0]
doc_lengths = np.asarray(X_skill.sum(axis=1)).ravel()
avgdl = doc_lengths.mean()

k1 = 1.5
b = 0.75
epsilon = 0.25

df = np.asarray((X_skill > 0).sum(axis=0)).ravel()

idf = np.log((N - df + 0.5) / (df + 0.5))

average_idf = idf.mean()
idf[idf < 0] = epsilon * average_idf

row_ids = np.repeat(
    np.arange(N),
    np.diff(X_skill.indptr)
)

norm = k1 * (
    1 - b + b * doc_lengths[row_ids] / avgdl
)

data = X_skill.data.astype(np.float32)

bm25_skill_values = (
    idf[X_skill.indices]
    * (data * (k1 + 1))
    / (data + norm)
)

bm25_skill_matrix = csr_matrix(
    (
        bm25_skill_values,
        X_skill.indices,
        X_skill.indptr
    ),
    shape=X_skill.shape,
    dtype=np.float32
)

resume_ids = resumes_df["resume_id"].tolist()

print("Skill-Aware BM25 ready.")
print("Resume matrix:", bm25_skill_matrix.shape)

Skill-Aware BM25 ready.
Resume matrix: (10000, 283)


In [14]:
jobs_eval["skill_aware_text"] = jobs_eval.apply(
    build_skill_aware_job_text,
    axis=1
)

print("skill_aware_text added to jobs_eval.")

skill_aware_text added to jobs_eval.


In [15]:
candidate_pools = {}

for _, job in jobs_eval.iterrows():

    query = skill_vectorizer.transform(
        [job["skill_aware_text"]]
    )

    scores = (
        bm25_skill_matrix @ query.T
    ).toarray().ravel()

    top_indices = np.argpartition(
        -scores, 100
    )[:100]

    top_indices = top_indices[
        np.argsort(-scores[top_indices])
    ]

    candidate_pools[job["job_id"]] = {
        "indices": top_indices,
        "resume_ids": [resume_ids[i] for i in top_indices],
        "scores": scores[top_indices]
    }

print("Candidate pools created:", len(candidate_pools))
print(
    "Candidates per job:",
    len(next(iter(candidate_pools.values()))["indices"])
)

Candidate pools created: 250
Candidates per job: 100


In [16]:
pool_recalls = []

for _, job in jobs_eval.iterrows():

    relevant_row = matches_df[
        matches_df["job_id"] == job["job_id"]
    ].iloc[0]

    relevant_ids = set(
        relevant_row["relevant_resume_ids"]
    )

    retrieved_ids = set(
        candidate_pools[job["job_id"]]["resume_ids"]
    )

    pool_recalls.append(
        len(retrieved_ids & relevant_ids)
        / len(relevant_ids)
    )

print(
    f"Skill-Aware BM25 Top-100 Recall: "
    f"{np.mean(pool_recalls):.6f}"
)

Skill-Aware BM25 Top-100 Recall: 0.108667


In [17]:
import time
import numpy as np

reranker_scores = {}

start_time = time.time()

for job_num, (_, job) in enumerate(jobs_eval.iterrows(), start=1):

    job_id = job["job_id"]

    candidate_resume_ids = candidate_pools[job_id]["resume_ids"]

    candidate_texts = (
        resumes_df
        .set_index("resume_id")
        .loc[candidate_resume_ids, "retrieval_text"]
        .tolist()
    )

    pairs = [
        (job["retrieval_text"], resume_text)
        for resume_text in candidate_texts
    ]

    scores = reranker_256.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    reranker_scores[job_id] = np.asarray(scores)

    if job_num % 25 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {job_num}/{len(jobs_eval)} jobs "
            f"| Elapsed: {elapsed / 60:.1f} min"
        )

total_time = time.time() - start_time

print("\nReranking complete.")
print(f"Jobs: {len(jobs_eval)}")
print(f"Pairs: {len(jobs_eval) * 100}")
print(f"Total time: {total_time / 60:.2f} min")

Processed 25/250 jobs | Elapsed: 2.0 min
Processed 50/250 jobs | Elapsed: 4.0 min
Processed 75/250 jobs | Elapsed: 6.0 min
Processed 100/250 jobs | Elapsed: 9.9 min
Processed 125/250 jobs | Elapsed: 11.9 min
Processed 150/250 jobs | Elapsed: 13.3 min
Processed 175/250 jobs | Elapsed: 14.7 min
Processed 200/250 jobs | Elapsed: 16.1 min
Processed 225/250 jobs | Elapsed: 17.6 min
Processed 250/250 jobs | Elapsed: 19.1 min

Reranking complete.
Jobs: 250
Pairs: 25000
Total time: 19.06 min


In [18]:
np.save(
    "data/reranker_250job_scores.npy",
    reranker_scores,
    allow_pickle=True
)

print("Reranker scores saved.")

Reranker scores saved.


In [19]:
import numpy as np

def recall_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)
    return len(set(ranked_ids[:k]) & relevant_ids) / len(relevant_ids)


def reciprocal_rank(ranked_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    for rank, resume_id in enumerate(ranked_ids, start=1):
        if resume_id in relevant_ids:
            return 1.0 / rank

    return 0.0


def ndcg_at_k(ranked_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    dcg = sum(
        1.0 / np.log2(i + 2)
        for i, resume_id in enumerate(ranked_ids[:k])
        if resume_id in relevant_ids
    )

    ideal_hits = min(len(relevant_ids), k)

    idcg = sum(
        1.0 / np.log2(i + 2)
        for i in range(ideal_hits)
    )

    return dcg / idcg if idcg > 0 else 0.0


metrics = [
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "mrr",
    "ndcg@5",
    "ndcg@10"
]

bm25_results = {m: [] for m in metrics}
reranked_results = {m: [] for m in metrics}

for _, job in jobs_eval.iterrows():

    job_id = job["job_id"]

    candidate_ids = candidate_pools[job_id]["resume_ids"]
    candidate_bm25_scores = candidate_pools[job_id]["scores"]
    candidate_reranker_scores = reranker_scores[job_id]

    # Original BM25 ordering
    bm25_ranked_ids = candidate_ids

    # Reranker ordering
    reranker_order = np.argsort(
        -candidate_reranker_scores
    )

    reranked_ids = [
        candidate_ids[i]
        for i in reranker_order
    ]

    # Ground-truth relevant resumes
    relevant_row = matches_df[
        matches_df["job_id"] == job_id
    ].iloc[0]

    relevant_ids = set(
        relevant_row["relevant_resume_ids"]
    )

    # BM25 metrics
    bm25_results["recall@1"].append(
        recall_at_k(bm25_ranked_ids, relevant_ids, 1)
    )
    bm25_results["recall@3"].append(
        recall_at_k(bm25_ranked_ids, relevant_ids, 3)
    )
    bm25_results["recall@5"].append(
        recall_at_k(bm25_ranked_ids, relevant_ids, 5)
    )
    bm25_results["recall@10"].append(
        recall_at_k(bm25_ranked_ids, relevant_ids, 10)
    )
    bm25_results["mrr"].append(
        reciprocal_rank(bm25_ranked_ids, relevant_ids)
    )
    bm25_results["ndcg@5"].append(
        ndcg_at_k(bm25_ranked_ids, relevant_ids, 5)
    )
    bm25_results["ndcg@10"].append(
        ndcg_at_k(bm25_ranked_ids, relevant_ids, 10)
    )

    # Reranked metrics
    reranked_results["recall@1"].append(
        recall_at_k(reranked_ids, relevant_ids, 1)
    )
    reranked_results["recall@3"].append(
        recall_at_k(reranked_ids, relevant_ids, 3)
    )
    reranked_results["recall@5"].append(
        recall_at_k(reranked_ids, relevant_ids, 5)
    )
    reranked_results["recall@10"].append(
        recall_at_k(reranked_ids, relevant_ids, 10)
    )
    reranked_results["mrr"].append(
        reciprocal_rank(reranked_ids, relevant_ids)
    )
    reranked_results["ndcg@5"].append(
        ndcg_at_k(reranked_ids, relevant_ids, 5)
    )
    reranked_results["ndcg@10"].append(
        ndcg_at_k(reranked_ids, relevant_ids, 10)
    )


bm25_metrics = {
    metric: np.mean(values)
    for metric, values in bm25_results.items()
}

reranked_metrics = {
    metric: np.mean(values)
    for metric, values in reranked_results.items()
}


print("BM25 → Top-K")
print("-" * 40)

for metric, value in bm25_metrics.items():
    print(f"{metric.upper():<12}: {value:.6f}")


print("\nQwen3 Reranker → Top-K")
print("-" * 40)

for metric, value in reranked_metrics.items():
    print(f"{metric.upper():<12}: {value:.6f}")


print("\nAbsolute Improvement")
print("-" * 40)

for metric in metrics:
    improvement = (
        reranked_metrics[metric]
        - bm25_metrics[metric]
    )

    print(f"{metric.upper():<12}: {improvement:+.6f}")

BM25 → Top-K
----------------------------------------
RECALL@1    : 0.001067
RECALL@3    : 0.004400
RECALL@5    : 0.006133
RECALL@10   : 0.011067
MRR         : 0.122549
NDCG@5      : 0.037063
NDCG@10     : 0.034593

Qwen3 Reranker → Top-K
----------------------------------------
RECALL@1    : 0.001333
RECALL@3    : 0.003067
RECALL@5    : 0.004800
RECALL@10   : 0.010533
MRR         : 0.115717
NDCG@5      : 0.030690
NDCG@10     : 0.031988

Absolute Improvement
----------------------------------------
RECALL@1    : +0.000267
RECALL@3    : -0.001333
RECALL@5    : -0.001333
RECALL@10   : -0.000533
MRR         : -0.006832
NDCG@5      : -0.006373
NDCG@10     : -0.002605


In [20]:
from sentence_transformers import CrossEncoder

reranker_jobmatch = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device="cuda",
    max_length=256,
    prompts={
        "job_matching": (
            "Judge whether the candidate resume provides relevant evidence "
            "for the job description. Use only information explicitly "
            "supported by the candidate resume."
        )
    },
    default_prompt_name="job_matching"
)

print("Job-matching reranker loaded.")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Default prompt name is set to 'job_matching'. This prompt will be applied to all inference calls, except if a `prompt` or `prompt_name` parameter is provided.


Job-matching reranker loaded.


In [21]:
import time

start = time.time()

sample_scores_jobmatch = reranker_jobmatch.predict(
    pairs,
    batch_size=16,
    show_progress_bar=True
)

elapsed = time.time() - start

print("Scores shape:", sample_scores_jobmatch.shape)
print(f"Time: {elapsed:.2f}s")
print(f"Pairs/sec: {100 / elapsed:.2f}")

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Scores shape: (100,)
Time: 19.88s
Pairs/sec: 5.03


In [22]:
from scipy.stats import spearmanr

corr, _ = spearmanr(
    sample_scores,
    sample_scores_jobmatch
)

top10_default = set(np.argsort(-sample_scores)[:10])
top10_jobmatch = set(np.argsort(-sample_scores_jobmatch)[:10])

overlap = len(top10_default & top10_jobmatch) / 10

print(f"Spearman correlation: {corr:.4f}")
print(f"Top-10 overlap: {overlap:.2%}")

Spearman correlation: -0.1435
Top-10 overlap: 10.00%


In [23]:
jobs_prompt_eval = (
    jobs_eval
    .iloc[:20]
    .copy()
    .reset_index(drop=True)
)

print("Prompt evaluation jobs:", len(jobs_prompt_eval))
print("Expected pairs:", len(jobs_prompt_eval) * 100)

Prompt evaluation jobs: 20
Expected pairs: 2000


In [24]:
prompt_test_scores = {}

start_time = time.time()

resume_lookup = resumes_df.set_index("resume_id")["retrieval_text"]

for job_num, (_, job) in enumerate(
    jobs_prompt_eval.iterrows(),
    start=1
):
    job_id = job["job_id"]

    candidate_ids = candidate_pools[job_id]["resume_ids"]

    candidate_texts = [
        resume_lookup[rid]
        for rid in candidate_ids
    ]

    pairs = [
        (job["retrieval_text"], resume_text)
        for resume_text in candidate_texts
    ]

    scores = reranker_jobmatch.predict(
        pairs,
        batch_size=16,
        show_progress_bar=False
    )

    prompt_test_scores[job_id] = np.asarray(scores)

    if job_num % 5 == 0:
        elapsed = time.time() - start_time
        print(
            f"Processed {job_num}/20 jobs "
            f"| {elapsed / 60:.1f} min"
        )

print("\nCustom-prompt test complete.")
print(f"Total time: {(time.time() - start_time) / 60:.2f} min")

Processed 5/20 jobs | 1.8 min
Processed 10/20 jobs | 3.6 min
Processed 15/20 jobs | 5.3 min
Processed 20/20 jobs | 7.2 min

Custom-prompt test complete.
Total time: 7.19 min


In [25]:
# Compare BM25, default Qwen reranker, and job-specific Qwen reranker
# on the same 20-job subset.

metrics = [
    "recall@1",
    "recall@3",
    "recall@5",
    "recall@10",
    "mrr",
    "ndcg@5",
    "ndcg@10"
]

def calculate_metrics(ranked_ids, relevant_ids):
    return {
        "recall@1": recall_at_k(ranked_ids, relevant_ids, 1),
        "recall@3": recall_at_k(ranked_ids, relevant_ids, 3),
        "recall@5": recall_at_k(ranked_ids, relevant_ids, 5),
        "recall@10": recall_at_k(ranked_ids, relevant_ids, 10),
        "mrr": reciprocal_rank(ranked_ids, relevant_ids),
        "ndcg@5": ndcg_at_k(ranked_ids, relevant_ids, 5),
        "ndcg@10": ndcg_at_k(ranked_ids, relevant_ids, 10),
    }


bm25_results_20 = {m: [] for m in metrics}
default_results_20 = {m: [] for m in metrics}
custom_results_20 = {m: [] for m in metrics}

for _, job in jobs_prompt_eval.iterrows():

    job_id = job["job_id"]

    candidate_ids = candidate_pools[job_id]["resume_ids"]

    # Ground truth
    relevant_row = matches_df[
        matches_df["job_id"] == job_id
    ].iloc[0]

    relevant_ids = set(
        relevant_row["relevant_resume_ids"]
    )

    # 1. Original Skill-Aware BM25 ranking
    bm25_ranked_ids = candidate_ids

    # 2. Default Qwen reranker
    default_scores = reranker_scores[job_id]
    default_order = np.argsort(-default_scores)

    default_ranked_ids = [
        candidate_ids[i]
        for i in default_order
    ]

    # 3. Custom job-matching prompt
    custom_scores = prompt_test_scores[job_id]
    custom_order = np.argsort(-custom_scores)

    custom_ranked_ids = [
        candidate_ids[i]
        for i in custom_order
    ]

    # Store metrics
    for metric, value in calculate_metrics(
        bm25_ranked_ids, relevant_ids
    ).items():
        bm25_results_20[metric].append(value)

    for metric, value in calculate_metrics(
        default_ranked_ids, relevant_ids
    ).items():
        default_results_20[metric].append(value)

    for metric, value in calculate_metrics(
        custom_ranked_ids, relevant_ids
    ).items():
        custom_results_20[metric].append(value)


# Aggregate
bm25_20 = {
    m: np.mean(v)
    for m, v in bm25_results_20.items()
}

default_20 = {
    m: np.mean(v)
    for m, v in default_results_20.items()
}

custom_20 = {
    m: np.mean(v)
    for m, v in custom_results_20.items()
}


print("20-Job Reranking Comparison")
print("=" * 72)
print(
    f"{'Metric':<12}"
    f"{'BM25':>14}"
    f"{'Default Qwen':>16}"
    f"{'Custom Prompt':>18}"
)

for metric in metrics:
    print(
        f"{metric.upper():<12}"
        f"{bm25_20[metric]:>14.6f}"
        f"{default_20[metric]:>16.6f}"
        f"{custom_20[metric]:>18.6f}"
    )

20-Job Reranking Comparison
Metric                BM25    Default Qwen     Custom Prompt
RECALL@1          0.001667        0.003333          0.000000
RECALL@3          0.003333        0.003333          0.003333
RECALL@5          0.005000        0.005000          0.006667
RECALL@10         0.010000        0.010000          0.008333
MRR               0.127385        0.165093          0.112145
NDCG@5            0.034961        0.041219          0.033785
NDCG@10           0.034195        0.037069          0.025844


## Reranking Conclusion

Qwen3-Reranker-0.6B was evaluated as a cross-encoder over the
Skill-Aware BM25 Top-100 candidate pool.

A 250-job development evaluation was performed using 25,000
job–resume pairs. The reranker did not consistently improve ranking
quality over the BM25 baseline and produced lower overall MRR and nDCG
on the 250-job evaluation.

A smaller 20-job prompt ablation showed different behavior, with the
default Qwen reranker improving MRR and nDCG relative to BM25. However,
the difference between the small and larger evaluations indicates
substantial subset sensitivity, so the 20-job result is not sufficient
to establish a general improvement.

The task-specific prompt also produced substantially different rankings
from the default configuration and did not improve the evaluated
metrics on the small prompt experiment.

Therefore, Qwen3-Reranker-0.6B is retained as an experimental baseline
but is not selected as the primary candidate-ranking component.

The project will instead focus on the core requirement-level task:
matching individual job requirements against resume evidence and
classifying the evidence as Strong, Partial, or Unsupported.